# โครงงานวิเคราะห์ข้อมูลด้วย Python
## ชุดข้อมูล Advertising: ความสัมพันธ์ระหว่างงบโฆษณา (TV, Radio, Newspaper) กับยอดขาย (Sales)

**วิธีใช้:** วางไฟล์ `Advertising.csv` ไว้โฟลเดอร์เดียวกับไฟล์นี้ แล้วกด Run All
กราฟทุกภาพจะถูกบันทึกลงโฟลเดอร์ `output` อัตโนมัติ (ใช้ส่งงานสัปดาห์ที่ 3 ได้เลย)

## 1. นำเข้า library และอ่านข้อมูล

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# สร้างโฟลเดอร์เก็บรูปกราฟ
os.makedirs('output', exist_ok=True)

df = pd.read_csv('Advertising.csv')
df = df.drop(columns='Unnamed: 0')   # คอลัมน์นี้เป็นแค่เลขลำดับ ไม่ใช่ข้อมูล
df.head()

## 2. ตั้งฟอนต์ภาษาไทยสำหรับกราฟ
ถ้าเครื่องไม่มี TH Sarabun New โปรแกรมจะลองฟอนต์ถัดไปในรายการให้เอง (Tahoma มากับ Windows และรองรับภาษาไทย)

In [ ]:
plt.rcParams['font.family'] = ['TH Sarabun New', 'Tahoma', 'Leelawadee UI', 'Noto Sans Thai', 'sans-serif']
plt.rcParams['font.size'] = 16
plt.rcParams['axes.unicode_minus'] = False   # แก้เครื่องหมายลบไม่แสดง

## 3. ตรวจสอบข้อมูลเบื้องต้น
ตอบ: ข้อมูลมีกี่แถว กี่คอลัมน์ ชนิดข้อมูลอะไร มีค่าสูญหายหรือแถวซ้ำหรือไม่

In [ ]:
print('จำนวนแถว, คอลัมน์ =', df.shape)
print()
df.info()

In [ ]:
print('ค่าสูญหายในแต่ละคอลัมน์')
print(df.isnull().sum())
print()
print('จำนวนแถวซ้ำ =', df.duplicated().sum())

## 4. สถิติเชิงพรรณนา
ค่าเฉลี่ยและส่วนเบี่ยงเบนมาตรฐานแบบตัวอย่าง (หารด้วย n-1 จึงใส่ `ddof=1`)

In [ ]:
summary = pd.DataFrame({
    'ค่าเฉลี่ย': df.mean(),
    'ส.บ.ม. (ddof=1)': df.std(ddof=1),
    'ต่ำสุด': df.min(),
    'มัธยฐาน': df.median(),
    'สูงสุด': df.max()
}).round(2)
summary

In [ ]:
# ดูสรุปทีเดียวด้วย describe() เพื่อเทียบผล
df.describe().round(2)

In [ ]:
# บันทึกตารางสรุปเป็นไฟล์ CSV สำหรับนำไปทำตารางในรายงาน (สัปดาห์ที่ 2)
summary.to_csv('output/ตารางสถิติ.csv', encoding='utf-8-sig')

## 5. ตรวจหาค่าผิดปกติด้วยกฎ 1.5 × IQR
ค่าที่ต่ำกว่า Q1 − 1.5×IQR หรือสูงกว่า Q3 + 1.5×IQR ถือเป็นค่าผิดปกติ

In [ ]:
rows = []
for col in df.columns:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    low = q1 - 1.5 * iqr
    high = q3 + 1.5 * iqr
    n = ((df[col] < low) | (df[col] > high)).sum()
    rows.append([col, q1, q3, iqr, low, high, n])

outlier_table = pd.DataFrame(rows, columns=['ตัวแปร', 'Q1', 'Q3', 'IQR', 'ขอบล่าง', 'ขอบบน', 'จำนวนค่าผิดปกติ']).round(2)
outlier_table

In [ ]:
# ดูแถวที่เป็นค่าผิดปกติของ Newspaper
q1 = df['Newspaper'].quantile(0.25)
q3 = df['Newspaper'].quantile(0.75)
iqr = q3 - q1
df[(df['Newspaper'] < q1 - 1.5*iqr) | (df['Newspaper'] > q3 + 1.5*iqr)]

## 6. กราฟ

### 6.1 ฮิสโตแกรม (ดูรูปร่างการกระจายของแต่ละตัวแปร)

In [ ]:
names = {'TV': 'โฆษณาทางทีวี (TV)', 'Radio': 'โฆษณาทางวิทยุ (Radio)',
         'Newspaper': 'โฆษณาทางหนังสือพิมพ์ (Newspaper)', 'Sales': 'ยอดขาย (Sales)'}

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for ax, col in zip(axes.flatten(), df.columns):
    ax.hist(df[col], bins=15, color='steelblue', edgecolor='white')
    ax.set_title('ฮิสโตแกรมของ ' + names[col])
    ax.set_xlabel(col)
    ax.set_ylabel('ความถี่')
plt.tight_layout()
plt.savefig('output/histogram.png', dpi=200)
plt.show()

### 6.2 แผนภาพกล่อง (ดูการกระจายและค่าผิดปกติ)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 5))
for ax, col in zip(axes, df.columns):
    ax.boxplot(df[col])
    ax.set_title(col)
    ax.set_xticks([])
fig.suptitle('แผนภาพกล่องของแต่ละตัวแปร')
plt.tight_layout()
plt.savefig('output/boxplot.png', dpi=200)
plt.show()

### 6.3 กราฟการกระจาย (ดูความสัมพันธ์ระหว่างงบโฆษณาแต่ละช่องทางกับยอดขาย)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, col in zip(axes, ['TV', 'Radio', 'Newspaper']):
    ax.scatter(df[col], df['Sales'], color='steelblue', alpha=0.7)
    r = df[col].corr(df['Sales'])
    ax.set_title(col + ' กับ Sales (r = ' + str(round(r, 3)) + ')')
    ax.set_xlabel('งบโฆษณา ' + col)
    ax.set_ylabel('Sales')
plt.tight_layout()
plt.savefig('output/scatter.png', dpi=200)
plt.show()

## 7. ค่าสหสัมพันธ์

In [ ]:
corr = df.corr().round(3)
corr

In [ ]:
# สหสัมพันธ์ของแต่ละช่องทางกับ Sales เรียงจากมากไปน้อย
print(corr['Sales'].drop('Sales').sort_values(ascending=False))
corr.to_csv('output/ตารางสหสัมพันธ์.csv', encoding='utf-8-sig')

## 8. แบ่งข้อมูล Train / Test และสร้างโมเดล Linear Regression
แบ่งข้อมูล 80% ไว้สอนโมเดล (train) และ 20% ไว้ทดสอบ (test) โดย `random_state=42` ทำให้แบ่งได้เหมือนเดิมทุกครั้ง
ตัวแปรต้น (X) คือ TV, Radio, Newspaper ตัวแปรตาม (y) คือ Sales

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

X = df[['TV', 'Radio', 'Newspaper']]
y = df['Sales']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
print('train =', X_tr.shape[0], 'แถว, test =', X_te.shape[0], 'แถว')

In [ ]:
model = LinearRegression()
model.fit(X_tr, y_tr)          # สอนด้วยข้อมูล train เท่านั้น

print('จุดตัด (intercept) =', round(model.intercept_, 4))
for name, c in zip(X.columns, model.coef_):
    print('สัมประสิทธิ์ของ', name, '=', round(c, 4))

In [ ]:
y_pred_tr = model.predict(X_tr)
y_pred_te = model.predict(X_te)

result = pd.DataFrame({
    'ชุดข้อมูล': ['train', 'test'],
    'R2': [r2_score(y_tr, y_pred_tr), r2_score(y_te, y_pred_te)],
    'RMSE': [np.sqrt(mean_squared_error(y_tr, y_pred_tr)), np.sqrt(mean_squared_error(y_te, y_pred_te))],
    'MAE': [mean_absolute_error(y_tr, y_pred_tr), mean_absolute_error(y_te, y_pred_te)]
}).round(4)
result.to_csv('output/ผลโมเดล.csv', index=False, encoding='utf-8-sig')
result

In [ ]:
# กราฟเทียบค่าจริงกับค่าที่โมเดลทำนาย (ชุด test)
plt.figure(figsize=(7, 6))
plt.scatter(y_te, y_pred_te, color='steelblue', alpha=0.8)
lo, hi = y.min(), y.max()
plt.plot([lo, hi], [lo, hi], color='red', linestyle='--')   # เส้นที่ทำนายแม่นยำพอดี
plt.title('ค่าจริงเทียบกับค่าทำนาย (ชุด test)')
plt.xlabel('Sales จริง')
plt.ylabel('Sales ที่โมเดลทำนาย')
plt.tight_layout()
plt.savefig('output/actual_vs_predicted.png', dpi=200)
plt.show()

## 9. สรุปผลและตีความ (เขียนเองจากผลที่ได้ด้านบน)

**แนวทางตอบ** (เติมตัวเลขจริงจากผลรันของพวกคุณ)

1. **สถิติเชิงพรรณนา:** ตัวแปรใดมีค่าเฉลี่ยสูงสุด และตัวแปรใดกระจายตัวมากที่สุด (ดูจากส่วนเบี่ยงเบนมาตรฐาน)
2. **รูปร่างการกระจาย:** ฮิสโตแกรมของแต่ละตัวแปรสมมาตร เบ้ขวา หรือเบ้ซ้าย
3. **ค่าผิดปกติ:** ตัวแปรใดมีค่าผิดปกติ กี่ค่า และมีผลต่อการวิเคราะห์อย่างไร
4. **ความสัมพันธ์:** ช่องทางโฆษณาใดสัมพันธ์กับยอดขายมากที่สุดและน้อยที่สุด ทิศทางเป็นบวกหรือลบ
5. **โมเดล:** ค่า R² ของชุด train และ test ใกล้เคียงกันหรือไม่ (ถ้า test ต่ำกว่ามาก แปลว่าโมเดลจำข้อมูลสอนมากเกินไป) และสัมประสิทธิ์ของช่องทางใดมากที่สุด
6. **ข้อควรระวัง:** สหสัมพันธ์ไม่ได้บอกว่าเป็นเหตุและผลกัน

*(พิมพ์สรุปของพวกคุณที่นี่)*